# Regresja liniowa

## Wstęp

Celem tego laboratorium będzie stworzenie modelu uczenia maszynowego do estymacji cen nieruchomości na podstawie danych o ich położeniu, liczbie pomieszczeń, roku budowy oraz wielu innych parametrów.

W trakcie realizacji tego laboratorium zapoznamy się z pełnym procesem uczenia maszynowego dla danych tabelarycznych, w szczególności z przygotowaniem danych oraz regresją liniową i jej regularyzowanymi wariantami.

**Uwagi:**
- niektóre zadania zamiast kodu wymagają podania pisemnej odpowiedzi w miejscu oznaczonym `// skomentuj tutaj`,
- pod zadaniami znajdują się asercje (`assert`), które pomogą Ci sprawdzić rozwiązanie; jeżeli nie udaje Ci się sprawić, żeby przechodziły, to napisz o tym komentarz.

## Wykorzystywane biblioteki

Na zajęciach korzystać będziesz z kilku popularnych bibliotek Pythona, które umożliwiają przetwarzanie danych, ich wizualizację czy tworzenie modeli. Są to:

* [NumPy](https://numpy.org/) - biblioteka do wykonywania wydajnych obliczeń macierzowych
* [Pandas](https://pandas.pydata.org/) - narzędzie do przetwarzania i analizy danych tabelarycznych
* [scikit-learn](https://scikit-learn.org/stable/) - standardowa biblioteka do tabelarycznego uczenia maszynowego w Pythonie
* [matplotlib](https://matplotlib.org/) - wykresy i wizualizacje, dobrze zintegrowana z Pandasem
* [feature-engine](https://feature-engine.trainindata.com/en/latest/index.html) - dodatkowe transformacje danych, kompatybilne ze scikit-learn

## Konfiguracja środowiska

Możesz korzystać ze środowiska lokalnego z menedżerem pakietów [uv](https://docs.astral.sh/uv/) lub z [Google Colab](https://colab.research.google.com/). Wszystko zostało przetestowane z użyciem `uv` - jeżeli w trakcie pracy instalujesz biblioteki ręcznie przez `pip`, to coś może się zepsuć lub nie działać z powodu niekompatybilnych wersji.

### Środowisko lokalne

Korzystamy z menedżera pakietów `uv`, który jest obecnie de facto standardem. Jest to aktualnie najpopularniejsze narzędzie do zarządzania środowiskami w Pythonie, szczególnie w przemyśle. Korzysta ze standardów zdefiniowanych w dokumentach PEP i jest zalecanym podejściem dla nowych projektów. Jest podobny do np. `cargo` w Ruście, `npm` w JavaScripcie czy `bundler` w Ruby.

W repozytorium znajduje się plik `pyproject.toml` ze zdefiniowanymi zależnościami (oraz ich ograniczeniami, np. `numpy>=2`). Na jego podstawie komenda `uv sync` wyznacza najnowsze wersje wszystkich bibliotek, kompatybilne ze sobą nawzajem, co nazywa się rozwiązywaniem zależności (*dependency resolution*). Jego wynikiem jest plik `uv.lock`, zawierający dokładną listę zależności, zarówno bezpośrednich, jak i przechodnich (*transitive*). Dzięki temu instalując zależności za pomocą `uv sync` dostaniemy dokładnie to samo środowisko na różnych komputerach, co maksymalizuje reprodukowalność. Jest to też znacznie szybsze niż zwykły `pip`, bo `uv` jest napisany w Ruście, dobrze zrównoleglony i ma wydajny cache.

Konfiguracja `uv` sprowadza się do następujących kroków:
1. Instalacja uv [zgodnie z dokumentacją](https://docs.astral.sh/uv/getting-started/installation/)
2. Stworzenie środowiska wirtualnego: `uv venv`
3. Aktywacja środowiska wirtualnego: zgodnie z komendą wypisaną przez `uv venv`, np. `source .venv/bin/activate` na Linuxie/macOS/WSL
4. Zainstalowanie zależności z pliku `uv.lock`: `uv sync`
5. Uruchomienie Jupyter Notebooka: `uv run jupyter notebook` (samo `jupyter notebook` też zadziała w aktywowanym venv'ie)

Jeżeli korzystasz z PyCharma, to ma on wbudowaną integrację z `uv`. Dla VS Code istnieją odpowiednie rozszerzenia.

### Google Colab

[Google Colab](https://colab.research.google.com/) to hostowane środowisko Jupyter Notebook. Udostępnia darmowe zasoby obliczeniowe, w tym mniejsze karty GPU. Może być całkiem wygodne w przypadku laboratoriów bardziej wymagających obliczeniowo, jeżeli nie masz własnego GPU.

Jeżeli korzystasz z tej opcji, to prześlij do katalogu `/content` oba pliki CSV dołączone do laboratorium ([ames_data.csv](ames_data.csv) oraz [melbourne_data.csv](melbourne_data.csv)), a także plik `requirements.txt`. Jest to zwykła lista zależności dla `pip`, ale została ona wyeksportowana bezpośrednio z pliku `uv.lock`, aby zapewnić reprodukowalność.

Aby zainstalować zależności, wykonaj `!pip install -r requirements.txt` w osobnej komórce notebooka. `!` oznacza wykonanie komendy basha w Jupyter Notebooku.

In [ ]:
# uncomment and run if using Google Colab
# !pip install -r requirements.txt

### Podstawowe importy

Zaimportujmy podstawowe biblioteki i przy okazji upewnijmy się, że środowisko działa.

Scikit-learn (główna biblioteka do ML w Pythonie) domyślnie zwraca tablice NumPy, tracąc nazwy cech z danych wejściowych. Ustawimy dlatego globalnie opcję `transform_output` ([dokumentacja](https://scikit-learn.org/stable/auto_examples/miscellaneous/plot_set_output.html)), aby zachować DataFrame'y z Pandasa z poprawnymi nazwami cech.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn

sklearn.set_config(transform_output="pandas")

## Zbiór danych Ames housing

Wykorzystamy zbiór danych [Ames housing](https://www.openintro.org/book/statdata/?data=ames), w którym zadaniem jest przewidywanie wartości domu na podstawie cech budynku, działki, lokalizacji itp. Jest to więc przewidywanie wartości ciągłej, czyli **regresja (regression)**. Zbiór ten zawiera zmienne numeryczne (float, int), kategoryczne nieuporządkowane (*categorical nominal*) oraz kategoryczne uporządkowane (*categorical ordinal*), więc będzie wymagał wstępnego przetworzenia, tak jak większość prawdziwych danych w uczeniu maszynowym.

Zbiór Ames housing został opisany oryginalnie w [tym artykule](https://jse.amstat.org/v19n3/decock.pdf) (Dean De Cock, 2011).

Wczytajmy dane z pliku [ames_data.csv](ames_data.csv) do zmiennej `df`. Typowo taką nazwą zmiennej oznaczamy obiekty `DataFrame` - zaawansowanej tablicy, np. z nazwami kolumn, używanej przez bibliotekę Pandas.

Mamy naprawdę dużo cech! Ich szczegółowy opis znajdziesz w dołączonym do laboratorium pliku [ames_description.txt](ames_description.txt).

In [ ]:
df = pd.read_csv("ames_data.csv")

# remove dots from names to match ames_description.txt
df.columns = [col.replace(".", "") for col in df.columns]

df.head()

### Wstępna analiza danych (EDA)

Zawsze, zanim zaczniesz robić jakąkolwiek predykcję czy analizę danych, dobrze jest zapoznać się z nimi, z ich kodowaniem i znaczeniem. Taki etap nazywa się **eksploracyjną analizą danych (exploratory data analysis, EDA)**. Sprawdzamy w nim m.in. liczbę próbek i cech, typy cech, wartości brakujące oraz rozkłady wartości. Typ danych jest szczególnie istotny - nie każdy model nadaje się do każdego typu.

Przydatne narzędzia w Pandas:
- `.shape` - liczba wierszy i kolumn
- `.info()` - nazwy kolumn, ich typy oraz liczba wartości niepustych
- `.describe()` - podsumowanie rozkładu kolumn numerycznych
- `.isna()` - maska wartości brakujących, zwykle w połączeniu z `.sum()` lub `.mean()`
- `.select_dtypes()` - wybiera kolumny według typu; `include="number"` daje kolumny numeryczne, a `exclude="number"` wszystkie pozostałe (napisy, czyli zmienne kategoryczne)

In [ ]:
df.info()

In [ ]:
# transpose for readability - features in rows, stats in columns
df.describe().T

### Zadanie 1 (0.5 punktu)

1. Oblicz liczbę próbek i cech, a następnie przypisz je do zmiennych `n_samples` oraz `n_features`. Pamiętaj, że `SalePrice` to nasza zmienna docelowa (labels), a nie cecha!
2. Oblicz liczbę cech numerycznych i kategorycznych, a następnie przypisz je do zmiennych `n_numerical` oraz `n_categorical`. Przyjmij, że wszystkie cechy nienumeryczne są kategoryczne.
3. Oblicz procent wartości brakujących dla każdej kolumny, jako liczby z zakresu $[0, 100]$. Zostaw tylko kolumny z wartościami brakującymi i posortuj je malejąco. Przypisz wynik do zmiennej `missing_pct` (jako `pd.Series`).
4. Narysuj `missing_pct` jako wykres słupkowy (*bar plot*). Pamiętaj o tytule i opisaniu osi.

Przydatne atrybuty i metody obiektu `DataFrame`: `.shape`, `.select_dtypes()`, `.isna()`, `.mean()`, `.mul()`, `.sort_values()`, `.plot.bar()`.

In [ ]:
# number of samples and features
# your_code

# number of numerical and categorical features
# your_code

# percentage of missing values
# your_code

# bar plot
# your_code

In [ ]:
assert n_samples == 2930
assert n_features == 81
assert n_numerical == 38
assert n_categorical == 43

assert len(missing_pct) == 27
assert missing_pct.is_monotonic_decreasing
assert 99 < missing_pct["PoolQC"] < 100

print("Solution is correct!")

Mamy sporo wartości brakujących, a niektóre cechy, jak `PoolQC` czy `Alley`, są nieobecne dla prawie wszystkich domów. Jeżeli jednak zajrzysz do pliku [ames_description.txt](ames_description.txt), to zobaczysz, że dla wielu cech "NA" oznacza po prostu "brak", np. "brak basenu". Zajmiemy się tym poniżej.

## Czyszczenie danych

### Usunięcie niepotrzebnych kolumn

Niektóre kolumny są **nieinformatywne (uninformative)**, czyli nie niosą żadnej informacji dla naszego zadania, czyli przewidywania wartości domu. Są pewnym rodzajem metadanych. Przykładowo mamy tutaj kolumny `Order` oraz `PID`.

`Order` jest po prostu numerem rekordu w zbiorze danych. Moglibyśmy przetasować cały zbiór i nie powinno to w żaden sposób wpłynąć na cokolwiek, a więc możemy spokojnie tę kolumnę usunąć.

Formalnie czynimy założenie, że rekordy w naszych danych (próbki / wiersze, poszczególne domy w przypadku tego zbioru) są **niezależne i pochodzą z tego samego rozkładu** (ang. **independent and identically distributed - i.i.d.**). Innymi słowy, kolejność w danych nie ma znaczenia, bo zbieraliśmy dane taką samą metodą i w identycznych warunkach. Jest to bardzo typowe w ML.

`PID` jest po prostu numerem identyfikacyjnym danej nieruchomości w systemie informatycznym, a więc też możemy go usunąć.

In [ ]:
# ignore errors so that you can run this cell many times (with features already removed)
df = df.drop(["Order", "PID"], axis="columns", errors="ignore")

### Usunięcie słabo reprezentowanych dzielnic

Dzielnice *GrnHill* oraz *Landmrk* obejmują w sumie zaledwie 3 domy, więc je usuwamy. Usuwanie rzadkich kategorii pomaga też uniknąć dużej wymiarowości po kodowaniu one-hot encoding, które omówimy później.

In [ ]:
df = df.loc[~df["Neighborhood"].isin(["GrnHill", "Landmrk"]), :]

### Usunięcie obserwacji odstających (outliers)

**Obserwacje odstające (outliers)** utrudniają modelowi nauczenie się typowych wzorców w danych, a regresja liniowa jest na nie dość wrażliwa. Autor zbioru zaleca usunięcie budynków, które mają powyżej 4000 stóp kwadratowych (ok. 370 metrów kwadratowych) powierzchni mieszkalnej. Jest ich dosłownie kilka, a niektóre z nich są bardzo duże, ale zaskakująco tanie. Takie skrajne przypadki raczej nas nie interesują - a na pewno stanowią problem dla tak prostego modelu jak regresja liniowa. Nie chcemy też, żeby nasz model uczył się takich anomalii, więc lepiej je usunąć.

Zawsze warto zrobić kilka wizualizacji, żeby wykryć takie przypadki. Możemy je zobaczyć na wykresie poniżej. Tutaj usuwamy je ręcznie, ale istnieją też algorytmy do detekcji i usuwania obserwacji odstających.

In [ ]:
ax = df.plot.scatter(
    x="GrLivArea", y="SalePrice", alpha=0.75, title="House area vs price"
)
plt.show()

### Transformacja logarytmiczna zmiennej zależnej

Zawsze warto też przyjrzeć się **rozkładowi zmiennej docelowej (target distribution)**, żeby poznać jej typ i skalę. Ceny mają zwykle **rozkład skośny (skewed distribution)**, co ma sens - mało jest bardzo drogich domów. Taki rozkład sprawia, że trening jest mniej stabilny.

Rozkład normalny jest zwykle korzystniejszy dla tworzenia modeli, bo daje sensowną "wartość środkową" do przewidywania, a także penalizuje tak samo błędy niezależnie od ich znaku (zaniżona i zawyżona predykcja). Dokonamy dlatego **transformacji logarytmicznej (log transform)**, czyli zlogarytmujemy zmienną docelową (zależną). Rozkład stanie się wtedy bliższy normalnemu. Używa się często `np.log1p`, czyli $\log(1 + x)$, a nie `np.log`, bo jest to stabilne numerycznie dla wartości bliskich zera (tutaj [wyjaśnienie](https://stackoverflow.com/questions/49538185/purpose-of-numpy-log1p)).

Dodatkowa korzyść z takiej transformacji jest taka, że regresja liniowa przewiduje dowolne wartości rzeczywiste. Po przekształceniu logarytmicznym jest to całkowicie ok, bo po odwróceniu transformacji predykcje zawsze będą dodatnie. W oryginalnej przestrzeni trzeba by natomiast wymusić przewidywanie tylko wartości dodatnich (ujemne ceny są bez sensu). Da się to zrobić, ale zwiększa to koszt obliczeniowy. Operowanie na tzw. log-price jest bardzo częste w finansach.

Modele wytrenowane w ten sposób przewidują "logarytmy dolarów", więc pamiętaj, żeby przed obliczeniem metryk odwrócić transformację za pomocą `np.expm1`.

### Zadanie 2 (0.5 punktu)

1. Usuń domy o powierzchni mieszkalnej (`GrLivArea`) większej niż 4000 stóp kwadratowych. Narysuj po tej operacji ten sam wykres punktowy (*scatter plot*), co powyżej.
2. Napisz funkcję `plot_distribution(values, title, bins=50)`, która rysuje histogram podanych wartości. Dodaj średnią i medianę jako pionowe przerywane linie, z ich wartościami w legendzie. Pamiętaj o tytule i opisaniu osi.
3. Narysuj rozkład zmiennej `SalePrice`.
4. Wyodrębnij kolumnę `SalePrice` do zmiennej `y`, stosując transformację logarytmiczną `np.log1p`. Pamiętaj o usunięciu jej z `df`. Narysuj rozkład `y`.

Przydatne metody:
- `.loc[]` do filtrowania wierszy
- `.pop()` do wyodrębnienia kolumny z `DataFrame`'a
- `.plot.hist()` do rysowania histogramów
- `ax.axvline()` do rysowania pionowych linii ([przykład](https://stackoverflow.com/a/57046310/9472066))

Podpowiedź: w Pandas korzysta się z `.loc[]` do filtrowania wierszy i kolumn. Pierwszy indeks oznacza, które wiersze zostawić, a drugi indeks, które kolumny wybrać. Jeżeli chcemy zostawić wszystko (np. nie usuwać żadnych kolumn), to zadziała standardowy Pythonowy `:`, jak przy indeksowaniu list.

In [ ]:
# remove outliers
# your_code


def plot_distribution(values: pd.Series, title: str, bins: int = 50) -> None:
    # your_code
    pass


# plot distributions and apply log transform
# your_code

In [ ]:
assert df["GrLivArea"].max() <= 4000
assert len(df) == 2922
assert "SalePrice" not in df.columns
assert len(y) == len(df)
assert 11 < y.mean() < 13

print("Solution is correct!")

### Uzupełnianie wartości brakujących

W ramach dalszego czyszczenia danych **uzupełnimy wartości brakujące (missing value imputation)**.

Trzeba tu jednak wziąć pod uwagę:
- zmienne kategoryczne - trzeba uzupełniać odpowiednio, np. modą
- wiele brakujących wartości - estymacja średnią czy medianą byłaby niedokładna, takie cechy często warto raczej usunąć w całości
- możliwość wykorzystania wiedzy o zmiennych na podstawie opisu cech

Można więc zastosować odpowiednią wiedzę i przyjąć wartości domyślne. Dla wielu cech opis mówi, że NA oznacza "brak", np. brak garażu. Uzupełniamy je stałymi: nową kategorią `"None"` dla cech kategorycznych oraz 0 dla cech numerycznych (np. powierzchni garażu). W przypadku zmiennych kategorycznych tworzymy w ten sposób nową wartość, która implicite reprezentuje wartość brakującą. Stałe nie korzystają z żadnych statystyk obliczonych na danych, więc możemy to bezpiecznie zrobić przed podziałem na zbiór treningowy i testowy.

W praktyce niestety zwykle nie jest tak łatwo, że mamy dokumentację, i ten krok zajmuje kilka godzin (lub dni) konsultacji z różnymi osobami w firmie, czyli ekspertami dziedzinowymi :) Czasami w ogóle nie da się ustalić, jaka wartość byłaby sensowna, ponieważ nie mamy żadnego dostępu do osób odpowiedzialnych za przygotowanie wykorzystywanego zbioru danych.

Poniższe wartości zostały zainspirowane [tym notebookiem na Kaggle](https://www.kaggle.com/code/juliencs/a-study-on-regression-applied-to-the-ames-dataset).

In [ ]:
# data description says NA means "not present", e.g. "no alley access", "no basement"
not_present_categorical = [
    "Alley",
    "BsmtQual",
    "BsmtCond",
    "BsmtExposure",
    "BsmtFinType1",
    "BsmtFinType2",
    "FireplaceQu",
    "GarageType",
    "GarageFinish",
    "GarageQual",
    "GarageCond",
    "PoolQC",
    "Fence",
    "MiscFeature",
    "MasVnrType",
]
for col in not_present_categorical:
    df[col] = df[col].fillna("None")

# no basement / garage / masonry veneer, so areas and counts are 0
not_present_numerical = [
    "BsmtFinSF1",
    "BsmtFinSF2",
    "BsmtUnfSF",
    "TotalBsmtSF",
    "BsmtFullBath",
    "BsmtHalfBath",
    "GarageCars",
    "GarageArea",
    "MasVnrArea",
]
for col in not_present_numerical:
    df[col] = df[col].fillna(0)

# a single missing value, we use the most common value (standard circuit breakers)
df["Electrical"] = df["Electrical"].fillna("SBrkr")

df.isna().sum().loc[lambda s: s > 0]

Dwie cechy dalej mają wartości brakujące:
1. `LotFrontage` - długość (w stopach) odcinka ulicy przylegającego do działki. Każda działka przylega do jakiejś ulicy, więc jest to prawdziwa wartość brakująca.
2. `GarageYrBlt` - rok budowy garażu. Brakuje go dla domów bez garażu i nie ma tu sensownej wartości stałej.

Uzupełnimy je automatycznie, czyli dokonamy imputacji, po podziale danych na zbiór treningowy i testowy.

W przypadku wykonywania tego typu zmian - o ile istnieje taka możliwość - warto rozważyć różne interpretacje brakujących wartości. Może okazać się, że przyjęte przez nas założenia są błędne i prowadzą do pogorszenia działania modelu. Dlatego często warto porównać jakość predykcji z danymi uzupełnionymi oraz z danymi, w których kolumna z brakującymi wartościami jest po prostu usuwana.

### Dane kategoryczne

Jak już zdążyliśmy zauważyć, istnieją dwa główne rodzaje danych: numeryczne (*numerical data*) oraz kategoryczne (*categorical data*). Ten podział jest bardzo istotny. Dane numeryczne to żadna niespodzianka, po prostu mają swoją wartość, jak np. `GrLivArea`, czyli powierzchnia mieszkalna budynku. Dane kategoryczne to takie, którym w większości przypadków nie można przyporządkować wartości liczbowej (pewnym wyjątkiem są dane kategoryczne uporządkowane - *categorical ordinal*).

Modele wymagają jednak liczb, więc zmienne trzeba zakodować numerycznie, a sposób kodowania zależy od ich typu:
- **binarne (binary)**, np. `CentralAir` o wartościach N/Y - po prostu 0 i 1
- **nominalne (nominal)**, czyli nieuporządkowane, np. `Neighborhood` - kodowanie kolejnymi liczbami całkowitymi narzuciłoby sztuczną kolejność, więc używamy kodowania one-hot encoding, opisanego dalej
- **porządkowe (ordinal)**, czyli uporządkowane, np. jakość od słabej do doskonałej - możemy przypisać im kolejne liczby całkowite, czyli zastosować **kodowanie porządkowe (ordinal encoding)**

Wyobraź sobie zmienną reprezentującą kolory o wartościach "red", "green" i "blue". Jeżeli zakodowałbyś je np. jako $red = 0$, $green = 1$, $blue = 2$, to stwierdzasz tym samym, że w pewnym sensie $red < green < blue$. Raczej nie ma powodu, żeby tak sądzić. Jest to zmienna, która ma skończoną liczbę wartości, ale są one nieuporządkowane. Taki typ to zmienne *categorical nominal*.

Szczególnym przypadkiem są zmienne binarne (*boolean*). Jest to u nas kolumna `CentralAir` (Central Air Conditioning). Z opisu w pliku [ames_description.txt](ames_description.txt) wiemy, że przyjmuje ona dokładnie dwie wartości kategoryczne: *N* (No) oraz *Y* (Yes). W takiej sytuacji wolno zakodować te wartości numerycznie jako 0 i 1. Stwierdzasz tym samym, że klimatyzacja albo jest, albo jej nie ma.

Sytuacją podobną, chociaż mniej oczywistą, może być zmienna `Street`, opisująca typ drogi wiodącej do nieruchomości. Jeśli znowu spojrzymy do opisu danych, to można zauważyć, że ta zmienna może przyjmować tylko dwie różne wartości - *Grvl* i *Pave*. I tu też możemy sobie pozwolić na zakodowanie tych wartości jako 0 i 1. Stwierdzamy wtedy, że droga jest *utwardzona* (Pave) dla wartości 1. Oczywiście równie dobrze można by zakodować to odwrotnie i stwierdzić, że droga jest *nieutwardzona* (Grvl), gdy wartość wynosi 1.

W Pandasie typy numeryczne są oparte o NumPy (np. `np.int64`), a zmienne kategoryczne, napisy itp. mają typ tekstowy (`str` od Pandas 3.0, wcześniej `object`).

Zmienne `MSSubClass` oraz `MoSold` są kategoryczne (tak wynika z informacji zawartej w pliku [ames_description.txt](ames_description.txt)), a są w naszych danych zapisane wprost jako liczby. Przekształćmy je zatem do poprawnego typu, czyli do napisów.

In [ ]:
df["MSSubClass"] = "SC" + df["MSSubClass"].astype(str)

months = [
    "Jan",
    "Feb",
    "Mar",
    "Apr",
    "May",
    "Jun",
    "Jul",
    "Aug",
    "Sep",
    "Oct",
    "Nov",
    "Dec",
]
month_to_int = dict(zip(range(1, 13), months))
df["MoSold"] = df["MoSold"].map(month_to_int)

Oprócz tego zakodujemy zmienne kategoryczne uporządkowane (*categorical ordinal*) z tekstowych na kolejne liczby całkowite. Wiele cech jest porządkowych, a kilka z nich korzysta z tej samej skali jakości.

Przykładowo zmienna `BsmtCond`, oceniająca stan piwnicy, ma następujące możliwe wartości:
* *NA* - No Basement (u nas już `None`)
* *Po* (Poor) - Severe cracking, settling, or wetness
* *Fa* (Fair) - dampness or some cracking or settling
* *TA* (Typical) - slight dampness allowed
* *Gd* (Good)
* *Ex* (Excellent)

Do takich wartości możemy dopasować pewną skalę punktową, bo są one naturalnie uporządkowane: None < Po < Fa < TA < Gd < Ex. Kodowanie porządkowe zachowuje kolejność i tworzy pojedynczą cechę zamiast wielu (jak w kodowaniu one-hot encoding).

Używamy tutaj zwykłych Pythonowych słowników i metody `.map()`, bo jest to bardzo czytelne przy wielu kolumnach. Scikit-learn ma do tego klasę [OrdinalEncoder](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OrdinalEncoder.html), w której kolejność wartości podaje się parametrem `categories`, np. `OrdinalEncoder(categories=[["Po", "Fa", "TA", "Gd", "Ex"]])`.

In [ ]:
quality_scale = {"None": 0, "Po": 1, "Fa": 2, "TA": 3, "Gd": 4, "Ex": 5}
quality_features = [
    "ExterQual",
    "ExterCond",
    "BsmtQual",
    "BsmtCond",
    "HeatingQC",
    "KitchenQual",
    "FireplaceQu",
    "GarageQual",
    "GarageCond",
]
for col in quality_features:
    df[col] = df[col].map(quality_scale)

ordinal_mappings = {
    "PoolQC": {"None": 0, "Fa": 1, "TA": 2, "Gd": 3, "Ex": 4},
    "BsmtExposure": {"None": 0, "No": 1, "Mn": 2, "Av": 3, "Gd": 4},
    "BsmtFinType1": {
        "None": 0,
        "Unf": 1,
        "LwQ": 2,
        "Rec": 3,
        "BLQ": 4,
        "ALQ": 5,
        "GLQ": 6,
    },
    "BsmtFinType2": {
        "None": 0,
        "Unf": 1,
        "LwQ": 2,
        "Rec": 3,
        "BLQ": 4,
        "ALQ": 5,
        "GLQ": 6,
    },
    "GarageFinish": {"None": 0, "Unf": 1, "RFn": 2, "Fin": 3},
    "Fence": {"None": 0, "MnWw": 1, "GdWo": 2, "MnPrv": 3, "GdPrv": 4},
    "Functional": {
        "Sal": 1,
        "Sev": 2,
        "Maj2": 3,
        "Maj1": 4,
        "Mod": 5,
        "Min2": 6,
        "Min1": 7,
        "Typ": 8,
    },
    "LandSlope": {"Sev": 1, "Mod": 2, "Gtl": 3},
    "LotShape": {"IR3": 1, "IR2": 2, "IR1": 3, "Reg": 4},
    "PavedDrive": {"N": 0, "P": 1, "Y": 2},
    "Utilities": {"ELO": 1, "NoSeWa": 2, "NoSewr": 3, "AllPub": 4},
}
for col, mapping in ordinal_mappings.items():
    df[col] = df[col].map(mapping)

# make sure that all values were mapped
assert df[quality_features + list(ordinal_mappings)].notna().all().all()

Wszystkie pozostałe kolumny tekstowe to zmienne kategoryczne nieuporządkowane. Wydzielmy sobie listy cech numerycznych i kategorycznych, co później ułatwi nam odwoływanie się do nich.

In [ ]:
categorical_features = df.select_dtypes(exclude="number").columns
numerical_features = df.select_dtypes(include="number").columns

print(f"Categorical features: {len(categorical_features)}")
print(f"Numerical features: {len(numerical_features)}")

## Podział na zbiór treningowy i testowy

Nasz zbiór podzielimy na dwa podzbiory: treningowy (75%) i testowy (25%). Zbiór treningowy pozwoli nam wytrenować model regresji liniowej, natomiast testowy - oszacować jego jakość.

Pamiętaj, że wyniki uzyskiwane przez model na danych treningowych nie odzwierciedlają tego, jak będzie on sobie radził na danych, których nie ma w zbiorze uczącym - do tego są one zawsze zbyt optymistyczne. Aby uzyskać taką informację, konieczne jest sprawdzenie, jak model radzi sobie na danych testowych. Daje nam to oszacowanie, jak dobrze model **generalizuje się (generalization)** na nowe dane.

Funkcja [train_test_split()](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html) przyjmuje:
- dowolną liczbę tablic o tej samej długości, np. macierz cech (*features*) `X` i wektor wartości docelowych (*targets*) `y`
- `test_size` - ułamek danych przeznaczony na zbiór testowy (lub dokładną liczbę próbek testowych, jeżeli podamy liczbę całkowitą)
- `random_state` - ziarno (*seed*) dla losowości

Można zauważyć, że zmienna `y` jest małą literą, natomiast `X_train_raw` czy `X_test_raw` są z dużej. Są to konwencje pochodzące z matematyki - macierze oznaczamy dużymi literami, a wektory małymi. W tym przypadku `y` to wektor etykiet (labels), a macierz `X` zawiera kolumny z cechami (features) i wiersze z próbkami (samples).

**Uwaga:** **zawsze** ustalaj na sztywno wartość parametru `random_state`, zarówno tutaj, jak i w modelach wykorzystujących losowość, żeby wyniki były reprodukowalne. [Dokumentacja scikit-learn](https://scikit-learn.org/stable/glossary.html#term-random_state) jest tutaj pomocna. W skrócie, dla stałej wartości wyniki na danej maszynie będą deterministyczne.

In [ ]:
from sklearn.model_selection import train_test_split

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    df, y, test_size=0.25, random_state=0
)

print(f"Training set size: {len(X_train_raw)}")
print(f"Test set size: {len(X_test_raw)}")

## Przetwarzanie danych z wykorzystaniem scikit-learn

Teraz trzeba dokonać transformacji naszych danych:
- skrajne wartości niektórych cech numerycznych trzeba przyciąć, czyli dokonać **winsoryzacji (winsorization)**
- zmienne numeryczne dalej mają wartości brakujące, więc trzeba je uzupełnić, inaczej **imputować (impute)**
- zmienne numeryczne trzeba przeskalować do zakresu wartości $[0, 1]$, czyli je **przeskalować (scale)** przez zastosowanie **min-max scaling**
- zmienne kategoryczne nieuporządkowane trzeba przetworzyć tak, aby model był w stanie je obsłużyć, czyli zakodować je za pomocą **one-hot encoding**

### API biblioteki scikit-learn

API scikit-learn opiera się na kilku zasadach:
1. Transformacje (preprocessing) i modele to klasy. Hiperparametry to argumenty konstruktora, wszystkie mają wartości domyślne.
2. `.fit(X, y)` uczy się z danych, np. statystyk do skalowania albo wag modelu. Wywołujemy ją **tylko na danych treningowych**.
3. `.transform(X)` zwraca przekształcone dane (tylko transformacje). Przekształca je za pomocą parametrów wyznaczonych w `.fit()`.
4. `.predict(X)` zwraca predykcje (tylko modele).
5. `.fit_transform(X)` to skrót dla `.fit()` + `.transform()`, więc jej też używamy tylko na danych treningowych - na zbiorze testowym wywołujemy już tylko `.transform()`. W przypadku niektórych transformacji wykorzystuje ich specyfikę i działa szybciej niż sekwencyjne wywołanie `.fit()` oraz `.transform()`.
6. Wartości wyznaczone w `.fit()` są zapisywane jako atrybuty z podkreślnikiem (*underscore*) na końcu, np. `.mean_` w `StandardScaler` albo `.coef_` w modelach liniowych.

Rozpatrzmy, jak działają poszczególne metody, na przykładzie klasy [StandardScaler](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html), która dokonuje standaryzacji cech. Do wykonania standaryzacji potrzebujemy dla każdej z cech określić 2 wartości - średnią $\mu$ oraz odchylenie standardowe $\sigma$. Formuła standaryzacji dla przypomnienia:

$$z = \frac{x - \mu}{\sigma}$$

```python
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
scaler.fit(X_train)  # computes mu and sigma for each feature

X_train = scaler.transform(X_train)  # scale features
X_test = scaler.transform(X_test)    # scale features in the same way
```

#### Metoda `.fit()`

Metodę `.fit()` wykonujemy tylko raz, dla **danych treningowych**. To powoduje, że obliczamy np. wartości $\mu$ oraz $\sigma$ dla każdej cechy, na podstawie wartości ze zbioru treningowego. Wyuczone wartości zostają zapisane w obiekcie `StandardScaler` i są później używane do przeprowadzenia standaryzacji zarówno dla danych treningowych, jak i testowych.

**Co, gdyby dla danych testowych przeprowadzić osobną standaryzację?**

Będziemy, na przykład, standaryzować kolumnę `GrLivArea` - powierzchnię mieszkalną. Załóżmy, że z danych treningowych wyszłoby, że średnia jest równa $60m^2$, a odchylenie standardowe - $20m^2$. Wtedy wartości z przedziału $[40, 80]$ zostaną przekształcone do $[-1, 1]$. Nasz model wykorzysta to przekształcenie i będzie uważał, że wartości po transformacji w pobliżu $0$ oznaczają domy średniej wielkości.

Wytrenowaliśmy model i dostajemy do predykcji kilkadziesiąt budynków z jakiejś zamożnej dzielnicy. Średnia powierzchnia dla tych budynków to około $160m^2$. Przeprowadzając osobno standaryzację dla takich danych testowych, zaburzylibyśmy rozkład tej cechy, gdyż tym razem wartości wokół $0$ oznaczałyby dość duże domy. Modele są niezwykle czułe na podobne zaburzenia - musimy przetwarzać dane spójnie, żeby nie doszło do takich sytuacji.

**Czemu nie wywołać `.fit()` na wszystkich danych, a nie tylko treningowych?**

Wydzieliliśmy dane testowe po to, żeby sprawdzać, jak model poradzi sobie z danymi, których do tej pory nigdy nie widział, bo to właśnie takie dane będzie on dostawać w praktyce, po wdrożeniu do realnego systemu. Ta ocena obejmuje też etap preprocessingu, w tym skalowania. Więc jeśli etap preprocessingu zobaczy dane testowe, to nie będziemy w stanie uczciwie estymować jego zachowania na nowych danych.

Wykorzystanie danych testowych w procesie treningu w jakikolwiek sposób to błąd **wycieku danych (data leakage)**. Skutkuje on niepoprawnym, nadmiernie optymistycznym oszacowaniem jakości modelu.

### Operacje przetwarzania danych

#### Winsoryzacja

Niektóre cechy numeryczne, np. powierzchnie, mają bardzo skośne rozkłady. Zamiast usuwać kolejne domy, przycinamy wartości, np. do 1. i 99. percentyla, obliczonych na danych treningowych. Wartości mniejsze od 1. percentyla zastępujemy jego wartością, a większe od 99. percentyla - wartością 99. percentyla. Dzięki temu nie tracimy żadnych danych, a skrajne wartości przestają dominować.

Scikit-learn nie ma do tego gotowej transformacji, ale istnieje wiele kompatybilnych z nim bibliotek. Wykorzystamy klasę [Winsorizer](https://feature-engine.trainindata.com/en/latest/user_guide/outliers/Winsorizer.html) z biblioteki [Feature-engine](https://feature-engine.trainindata.com/en/latest/index.html). Jej transformacje działają na DataFrame'ach, a kolumny do przekształcenia wybiera się parametrem `variables`.

#### Imputacja brakujących wartości numerycznych

Wcześniej już napotkaliśmy wartości brakujące i postaraliśmy się uzupełnić je jak najlepiej potrafiliśmy, używając dokumentacji naszego zbioru. Nie gwarantuje to jednak usunięcia wszystkich braków. Nie zawsze w praktyce da się też tak łatwo znaleźć wartości do uzupełnienia. W przypadku zwykłych cech numerycznych możemy zastosować jedną z kilku bardzo popularnych strategii radzenia sobie z wartościami brakującymi:

1. Usunąć kolumnę, która zawiera brakujące wartości.
1. Usunąć wiersze, w których brakuje wartości.
1. Zastąpić brakujące wartości innymi, np. średnią z kolumny, medianą albo wartością stałą.
1. Przewidzieć brakujące wartości, wykorzystując odpowiedni model uczenia maszynowego.

Podejście 4 jest często zbyt czasochłonne. Opcje 1 i 2 prowadzą do utraty danych. My wypróbujemy sposób nr 3 i uzupełnimy wartości brakujące średnią za pomocą klasy [SimpleImputer](https://scikit-learn.org/stable/modules/generated/sklearn.impute.SimpleImputer.html) (`strategy="mean"`; inne opcje to `"median"`, `"most_frequent"` oraz `"constant"`).

Nie znaczy to jednak, że usunięcie wierszy czy kolumny jest zawsze złym podejściem. Usunięcie kolumny jest uzasadnione, jeśli ma ona naprawdę dużo wartości brakujących. W takich wypadkach ciężko z niej wyciągnąć jakąkolwiek sensowną informację. Usunięcie wierszy może być uzasadnione w przypadku, gdy mamy dużo rekordów i tylko niewielka część z nich posiada wartości brakujące (usunięcie kilku wierszy nie powinno powodować problemu).

#### Skalowanie

Jest to bardzo ważny krok dla wielu modeli sztucznej inteligencji. Często takie modele mają pewne założenia co do danych wejściowych, a szczególnie popularnym założeniem jest, że wszystkie cechy mają wartości o podobnej skali. W szczególności modele liniowe z regularyzacją (omówione dalej) wymagają cech o podobnej skali, bo w przeciwnym wypadku kara za wielkość wag zależy od jednostek cech. Dlatego trzeba przeskalować nasze dane, żeby spełnić to założenie.

Najprostsza metoda to [MinMaxScaler](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.MinMaxScaler.html), który przekształca wszystkie wartości do przedziału $[0, 1]$. Istnieją też inne metody, np. standaryzacja ([StandardScaler](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html)), którą możesz pamiętać ze statystyki (jej wynikiem jest Z-score). Polega na odjęciu średniej i podzieleniu przez odchylenie standardowe każdej cechy. Wynikiem przekształcenia są cechy o średniej 0 i odchyleniu standardowym 1.

Więcej informacji na temat tego, dlaczego skalowanie jest tak istotne, możesz znaleźć [tutaj](https://analyticsindiamag.com/why-data-scaling-is-important-in-machine-learning-how-to-effectively-do-it/).

#### Kodowanie one-hot encoding

Powyżej omawialiśmy zmienne kategoryczne. Typ *categorical ordinal* można zakodować kolejnymi liczbami całkowitymi, co jest oczywiście proste. Co jednak ze zmiennymi bez kolejności, typu *categorical nominal*? Trzeba je dalej przekształcić na liczby (żeby model był w stanie je przetworzyć), ale tak, aby nie nadać im implicite kolejności.

Spójrzmy na kolumnę `Neighborhood`, oznaczającą poszczególne dzielnice. Dom znajduje się tylko w jednej dzielnicy, a w pozostałych go nie ma. Idea kodowania **one-hot encoding** polega na stworzeniu tylu zmiennych binarnych, ile jest możliwych wartości, a następnie w każdym wierszu przypisaniu wartości 1 w tej kolumnie, która odpowiada oryginalnej wartości zmiennej.

Przykładowo, jeżeli mielibyśmy 3 wartości `["A", "B", "C"]`, to powstają z nich 3 cechy (kolumny macierzy `X`) `[col_A, col_B, col_C]`. Wiersz z pierwotną wartością `"B"` będzie miał wartości tych cech `[0, 1, 0]`. W przypadku naszej zmiennej `Neighborhood` pojawią się osobne zmienne `Neighborhood_OldTown`, `Neighborhood_NoRidge` itd., a dla każdego wiersza dokładnie jedna z nich będzie miała wartość 1.

W scikit-learn implementuje to klasa [OneHotEncoder](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OneHotEncoder.html). Ważne parametry:
- `drop="first"` - usuwa jedną nadmiarową cechę dla każdej zmiennej ($K-1$ zamiast $K$ cech); wszystkie $K$ cech zawsze sumuje się do 1, więc są one **idealnie współliniowe (perfectly collinear)** z wyrazem wolnym (intercept), co jest niestabilne numerycznie dla regresji liniowej
- `handle_unknown="ignore"` - jako że przekształcamy dane już po podziale na zbiór treningowy i testowy, to możemy spotkać na zbiorze testowym nieliczne przypadki kategorii, których nie ma w zbiorze treningowym; kodujemy je wtedy po prostu jako wektory zer
- `sparse_output=False` - zwraca zwykłe tablice zamiast macierzy rzadkich

In [ ]:
from feature_engine.outliers import Winsorizer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder

# skewed features with extreme values, for winsorization
skewed_features = [
    "LotFrontage",
    "LotArea",
    "MasVnrArea",
    "BsmtFinSF1",
    "TotalBsmtSF",
    "GarageArea",
    "WoodDeckSF",
    "OpenPorchSF",
]

X_train_num_raw = X_train_raw[numerical_features]
X_test_num_raw = X_test_raw[numerical_features]

X_train_cat_raw = X_train_raw[categorical_features]
X_test_cat_raw = X_test_raw[categorical_features]

### Zadanie 3 (1.5 punktu)

Dokonaj przetworzenia danych treningowych i testowych. Cechy numeryczne i kategoryczne zostały już rozdzielone w komórce powyżej.

1. Winsoryzacja: stwórz obiekt [Winsorizer](https://feature-engine.trainindata.com/en/latest/api_doc/outliers/Winsorizer.html) w zmiennej `winsorizer`, przycinający cechy z listy `skewed_features` do 1. i 99. percentyla. Użyj parametrów `capping_method="quantiles"`, `tail="both"` oraz `fold=0.01`. `LotFrontage` dalej ma wartości brakujące, więc przekaż też `missing_values="ignore"`. Wytrenuj go na treningowych cechach numerycznych, a następnie przetransformuj zarówno treningowe, jak i testowe cechy numeryczne.
2. Cechy numeryczne: uzupełnij wartości brakujące średnią ([SimpleImputer](https://scikit-learn.org/stable/modules/generated/sklearn.impute.SimpleImputer.html)), a następnie przeskaluj cechy do zakresu $[0, 1]$ ([MinMaxScaler](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.MinMaxScaler.html)).
3. Cechy kategoryczne: zastosuj kodowanie one-hot encoding ([OneHotEncoder](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OneHotEncoder.html)) z parametrami opisanymi powyżej.
4. Połącz wyniki dla cech numerycznych i kategorycznych w `X_train` oraz `X_test`, za pomocą `pd.concat()` wzdłuż kolumn.

Pamiętaj, żeby wywoływać `.fit()` lub `.fit_transform()` tylko na danych treningowych!

Prawdopodobnie zobaczysz ostrzeżenie o nieznanych kategoriach znalezionych podczas transformacji (*unknown categories found during transform*). Jest to oczekiwane, tak jak opisaliśmy powyżej dla `handle_unknown="ignore"`.

Pamiętaj, że zmienne `categorical_features` oraz `numerical_features` zdefiniowaliśmy powyżej.

In [ ]:
# winsorization
# your_code

# numerical features
# your_code

# categorical features
# your_code

# combine features
# your_code

In [ ]:
assert X_train.shape == (2191, 230)
assert X_test.shape == (731, 230)

assert X_train.notna().all().all()
assert X_test.notna().all().all()
assert X_train.min().min() >= -1e-6
assert X_train.max().max() <= 1 + 1e-6

assert winsorizer.left_tail_caps_["LotArea"] > 1000
assert winsorizer.right_tail_caps_["LotArea"] < 30000

print("Solution is correct!")

Zobaczmy, co winsoryzacja zrobiła z rozkładem cechy `LotArea`.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

df["LotArea"].plot.hist(bins=50, ax=axes[0], color="blue")
axes[0].set_title("LotArea, original")
axes[0].set_xlabel("LotArea")

winsorizer.transform(X_train_num_raw)["LotArea"].plot.hist(
    bins=50, ax=axes[1], color="blue"
)
axes[1].set_title("LotArea, after winsorization (training data)")
axes[1].set_xlabel("LotArea")

plt.show()

Nasze dane są teraz w pełni przetworzone. Dzięki ustawieniu na początku notebooka `transform_output="pandas"` dalej mamy czytelne nazwy cech, np. `Neighborhood_OldTown` dla cech po kodowaniu one-hot encoding.

In [ ]:
X_train.head()

## Regresja liniowa

Możemy teraz przejść do przewidywania wartości domów! Wykorzystamy do tego regresję liniową i jej warianty z regularyzacją.

### Regresja liniowa w scikit-learn

#### Model

Naszym narzędziem będzie tutaj **regresja liniowa (linear regression)**, czyli model postaci:
$$
\hat{y} = w^Tx = w_0 + \sum_{i=1}^{D} w_i x_i
$$
gdzie:
- $\hat{y}$ - wartość przewidywana
- $D$ - **wymiarowość (dimensionality)**, czyli liczba cech
- $x$ - wektor wartości cech, długości $D$
- $w$ - wektor **wag (weights)**, długości $D$
- $w_i$ - waga $i$-tej cechy
- $w_0$ - **wyraz wolny (intercept / bias)**, czyli punkt przecięcia z osią $y$; można sobie go wyobrazić jako wyrazu dla sztucznej, dodatkowej cechy o stałej wartości 1, tj. $x_0 = 1$

Wagi $w$ wyznaczane są przez minimalizację **błędu średniokwadratowego (mean squared error, MSE)** na zbiorze treningowym. Jest to przykład **funkcji kosztu (loss function / cost function)**. Ma ona postać:
$$
MSE(y, \hat{y}) = \frac{1}{N} \sum_{i=1}^N \left( y_i - \hat{y}_i \right)^2 = \frac{1}{N} ||y - \hat{y}||^2
$$
gdzie $\hat{y}_i$ to wartość przewidywana przez model dla $i$-tego punktu, $y_i$ - wartość prawdziwa, a $N$ to liczba punktów w zbiorze.

W scikit-learn ten model implementuje klasa [LinearRegression](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html). Jej ważne cechy:
- wagi są obliczane dokładnie, z użyciem pseudoodwrotności Moore'a-Penrose'a (SVD), więc nie ma żadnych hiperparametrów do dobrania
- nie pozwala na regularyzację, do tego trzeba użyć innych klas
- po wywołaniu `.fit()` wagi znajdują się w atrybucie `.coef_`, a wyraz wolny w `.intercept_`

```python
from sklearn.linear_model import LinearRegression

lin_reg = LinearRegression()
lin_reg.fit(X_train, y_train)

y_pred = lin_reg.predict(X_test)
```


#### Metryki w regresji

Jak ocenić, jak taki model sobie radzi? Trzeba tutaj użyć pewnej **metryki (metric)**, czyli wyznacznika jakości modelu. Można na to patrzeć z wielu różnych perspektyw, w zależności od charakterystyki problemu. Wykorzystamy dwie metryki, obie wyrażone w jednostce zmiennej docelowej (dolarach):
- **pierwiastek z błędu średniokwadratowego (root mean squared error, RMSE)** $= \sqrt{\frac{1}{N} \sum_{i=1}^N (y_i - \hat{y}_i)^2}$
- **średni błąd bezwzględny (mean absolute error, MAE)** $= \frac{1}{N} \sum_{i=1}^N |y_i - \hat{y}_i|$

RMSE mocno penalizuje duże błędy, np. dla bardzo drogich domów, więc jest dość czułe na wartości odstające. Może to być korzystne lub nie, w zależności od zastosowania. W MAE wszystkie błędy liczą się tak samo, więc lepiej opisuje ono jakość dla typowych domów.

W scikit-learn są to funkcje [root_mean_squared_error()](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.root_mean_squared_error.html) oraz [mean_absolute_error()](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.mean_absolute_error.html) z modułu `sklearn.metrics`, przyjmujące argumenty `(y_true, y_pred)`. Pamiętaj, żeby najpierw odwrócić transformację logarytmiczną za pomocą `np.expm1`, aby dostać błędy w dolarach, a nie "log-dolarach".

#### Przeuczenie i niedouczenie

W trakcie trenowania modelu może dojść do sytuacji, w której zostanie on **przeuczony (overfitting)**. W takim wypadku model nadmiernie dostosowuje się do danych treningowych, "zakuwając" je. Daje wtedy bardzo dokładne wyniki na zbiorze treningowym, ale kiepskie na zbiorze testowym. Modele przeuczone słabo zatem się **generalizują (generalization)**.

Dlatego wcześniej wydzieliliśmy zbiór testowy, za pomocą którego oceniamy skuteczność naszego modelu. Przeuczenie bardzo często można rozpoznać właśnie po różnym zachowaniu modelu na danych treningowych i testowych. Jeśli z danymi treningowymi model radzi sobie dużo lepiej niż z testowymi, to istnieje duże ryzyko, że model został przeuczony i skupił się na zapamiętywaniu konkretnych przykładów, na których się uczył, zamiast na wyciąganiu z nich uniwersalnych wzorców. Taki model słabo się generalizuje i nie poradzi sobie z nowymi danymi.

Sprawdza się to następująco, porównując błąd treningowy i testowy:
- oba błędy są wysokie - mamy zbyt małe dopasowanie, czyli **niedouczenie (underfitting)**; model jest zbyt prosty i trzeba użyć modelu o większej pojemności
- błąd treningowy jest dużo niższy od testowego - mamy nadmierne dopasowanie, czyli przeuczenie (*overfitting*); model "zakuł" dane treningowe i trzeba go regularyzować
- oba błędy są niskie i podobne - model jest dobrze dopasowany

To, co oznacza "wysoki" błąd albo "dużo niższy", zależy od problemu i skali zmiennej docelowej, więc zawsze wymaga to pewnej oceny z naszej strony.

### Zadanie 4 (1 punkt)

1. Zaimplementuj funkcję `evaluate_model()`:
    - oblicz predykcje dla zbioru treningowego i testowego,
    - zastosuj transformację odwrotną `np.expm1` do wartości docelowych i predykcji, żeby wrócić do oryginalnej jednostki (dolarów),
    - oblicz RMSE oraz MAE dla zbioru treningowego i testowego,
    - wypisz te metryki, zaokrąglone do 2 miejsc po przecinku,
    - zwróć metryki jako słownik z kluczami `"train_rmse"`, `"test_rmse"`, `"train_mae"` oraz `"test_mae"`.
2. Wytrenuj model `LinearRegression` w zmiennej `reg_linear` i dokonaj jego ewaluacji. Zapisz wyniki w słowniku `results`, pod kluczem `"Linear regression"`. Wykorzystamy ten słownik później do porównania wszystkich modeli.

Skomentuj wyniki. Czy następuje przeuczenie modelu? Oceń także sam błąd - czy subiektywnie to duża wartość, biorąc pod uwagę rozkład zmiennej docelowej (patrz histogram cen powyżej)?

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, root_mean_squared_error


def evaluate_model(model, X_train, X_test, y_train, y_test) -> dict[str, float]:
    # predict for train and test
    # your_code

    # inverse transform for targets and predictions
    # your_code

    # calculate train and test RMSE and MAE
    # your_code

    # print metrics
    # your_code

    # return metrics
    # your_code
    pass


results = {}

# train and evaluate linear regression
# your_code

In [ ]:
assert set(results["Linear regression"].keys()) == {
    "train_rmse",
    "test_rmse",
    "train_mae",
    "test_mae",
}

assert 16000 < results["Linear regression"]["train_rmse"] < 17500
assert 21000 < results["Linear regression"]["test_rmse"] < 22500
assert 11000 < results["Linear regression"]["train_mae"] < 12000
assert 12500 < results["Linear regression"]["test_mae"] < 13500

print("Solution is correct!")

// skomentuj tutaj

## Analiza błędów

Widać wyraźną różnicę między RMSE a MAE. Żeby ją zrozumieć, możemy przyjrzeć się rozkładowi **rezyduów (residuals)**, czyli błędów ze znakiem $y - \hat{y}$. Dodatnie rezydua oznaczają, że zaniżyliśmy wartość domu, a ujemne - że ją zawyżyliśmy.

Wykresy są tu bardzo przydatne. Metryki to tylko pojedyncze liczby, natomiast wykresy mogą nam pokazać np. czy popełniamy kilka bardzo dużych błędów, czy dużo średnich, i dla jakich domów.

### Zadanie 5 (1 punkt)

1. Oblicz rezydua modelu regresji liniowej na zbiorze testowym, w dolarach. Przypisz je do zmiennej `residuals`.
2. Narysuj rozkład rezyduów za pomocą `plot_distribution()`. Dobierz liczbę przedziałów (*bins*) tak, żeby wykres był czytelny.
3. Narysuj wykres punktowy prawdziwych cen (oś X) względem przewidywanych cen (oś Y) na zbiorze testowym, w dolarach. Dodaj przekątną $y = x$, która reprezentuje idealne predykcje. Pamiętaj o tytule, opisaniu osi i legendzie.

Na podstawie wykresów skomentuj rozkład błędów:
- czy rozkład rezyduów jest symetryczny?
- czy częściej zaniżamy, czy zawyżamy ceny?
- czy występują wartości odstające?
- czy na podstawie tych wykresów potrafisz wyjaśnić, czemu RMSE jest większe od MAE?

In [ ]:
# your_code

In [ ]:
assert len(residuals) == len(y_test)
assert abs(residuals.mean()) < 2000
assert 110000 < residuals.max() < 130000

print("Solution is correct!")

// skomentuj tutaj

## Regresja z regularyzacją (ridge, LASSO, ElasticNet)

Przeuczenie modelu jest bardzo istotnym problemem w sztucznej inteligencji i istnieje szereg metod służących zapobieganiu temu zjawisku. Główną jest **regularyzacja (regularization)** - do funkcji kosztu dodawane są "kary" za tworzenie zbyt złożonych modeli. Typowe metody regularyzacji to L1 oraz L2, które penalizują wielkość parametrów obliczonych w trakcie treningu. Obie te kary są tak naprawdę normami (odpowiednio L1 i L2) wektora wag modelu, przeskalowanymi przez określoną wartość - **siłę regularyzacji (regularization strength)**. Zmniejsza to pojemność modelu, a dodawanie tych kar ma zapobiec przeuczeniu, bo typowo duże wagi w regresji liniowej i podobnych modelach oznaczają przeuczenie.

Czemu tak jest? Przeuczenie bierze się z tego, że nasz model "zakuwa" zbiór treningowy, ucząc się **szumu (noise)** w danych, przypisując nadmierne znaczenie niewielkim różnicom w wartościach cech. Jeżeli cecha ma dużą wagę, to nawet niewielka zmiana jej wartości bardzo zmienia finalną predykcję (która jest kombinacją liniową). Dzięki regularyzacji, jeżeli model podczas treningu będzie chciał zwiększyć wagę dla cechy, to musi mu się to opłacać. Innymi słowy, zwiększenie wagi cechy musi zmniejszyć koszt (np. MSE) bardziej, niż wzrośnie kara z regularyzacji.

Zbyt duże kary spowodują z kolei niedouczenie (*underfitting*), narzucając na model zbyt silne ograniczenia.

W praktyce paradoksalnie często model o większej pojemności z mocną regularyzacją działa lepiej od prostszego modelu ze słabą regularyzacją. Wyjaśnienie, czemu tak jest, to otwarty problem naukowy, szczególnie w kontekście sieci neuronowych.

Regresja liniowa z regularyzacją L2 nazywa się **ridge regression**, z regularyzacją L1 - **LASSO regression**, a z oboma naraz - **ElasticNet regression**. Formalnie mamy:
$$
L_{ridge}(y, \hat{y}) = MSE(y, \hat{y}) + \lambda ||w||^2
$$
$$
L_{LASSO}(y, \hat{y}) = MSE(y, \hat{y}) + \alpha ||w||_1
$$
$$
L_{ElasticNet}(y, \hat{y}) = MSE(y, \hat{y}) + \alpha ||w||_1 + \lambda ||w||^2
$$

1. **Regresja ridge** - kara L2 silnie zmniejsza wagi; jest różniczkowalna, więc model jest szybki i łatwy w treningu
2. **Regresja LASSO** - kara L1 zmniejsza wagi, a przy tym często ustawia wiele z nich dokładnie na zero, eliminując słabe cechy; dokonuje w ten sposób **selekcji cech (feature selection)**
3. **ElasticNet** - obie kary naraz

Jak widać, regularyzacja dodaje do zwykłego kosztu MSE dodatkowe wyrazy, penalizujące wielkość wag $w$. **Siłę regularyzacji (regularization strength)**, czyli jak mocna jest taka kara, wyznacza współczynnik, oznaczany typowo $\lambda$ albo $\alpha$. Jest to **hiperparametr (hyperparameter)**, czyli stała modelu, którą narzucamy z góry, przed treningiem. Nie jest on uczony z danych. Jak go dobrać, omówimy poniżej.

W scikit-learn implementują je klasy [Ridge](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Ridge.html), [Lasso](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Lasso.html) oraz [ElasticNet](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.ElasticNet.html). Ważne parametry:
- `alpha` - siła regularyzacji (domyślnie 1.0), we wszystkich trzech klasach
- `l1_ratio` - ułamek $\rho$ siły regularyzacji, który przypada na karę L1 w ElasticNet (na karę L2 przypada $1 - \rho$)
- `max_iter` - maksymalna liczba iteracji solwera dla LASSO i ElasticNet; im więcej, tym dokładniejsze rozwiązanie, ale tym dłuższy czas obliczeń; jeżeli dostaniesz ostrzeżenie o braku zbieżności (*convergence warning*), to zwykle trzeba ją po prostu zwiększyć, np. 10-krotnie
- `tol` - tolerancja solwera; jeżeli nie potrzebujemy bardzo precyzyjnego rozwiązania, można ją zwiększyć dla przyspieszenia obliczeń (mało istotne w praktyce)

Sprawdźmy te modele z domyślnymi hiperparametrami.

In [ ]:
from sklearn.linear_model import Lasso, Ridge

reg_ridge = Ridge()
reg_ridge.fit(X_train, y_train)

reg_lasso = Lasso(random_state=0)
reg_lasso.fit(X_train, y_train)

print("Ridge")
evaluate_model(reg_ridge, X_train, X_test, y_train, y_test)
print()
print("LASSO")
evaluate_model(reg_lasso, X_train, X_test, y_train, y_test);

W przypadku regularyzacji L2 domyślna siła regularyzacji (`alpha=1.0`) znacząco poprawiła błąd testowy. Natomiast LASSO działa naprawdę źle! Przeprowadźmy **diagnostykę modelu (model diagnostics)**, żeby zobaczyć, co się dzieje. W takich przypadkach warto sprawdzić:
- predykcje modelu
- wagi (parametry, współczynniki)
- hiperparametry

W modelach liniowych wagi znajdują się w atrybutach `.coef_` oraz `.intercept_`. Atrybuty z `_` (underscore) na końcu w scikit-learn oznaczają, że zostały one wyliczone podczas treningu (w metodzie `.fit()`).

In [ ]:
print(f"Predictions: {np.expm1(reg_lasso.predict(X_test[:5]))}")
print(f"Non-zero weights: {np.sum(reg_lasso.coef_ != 0)} / {len(reg_lasso.coef_)}")
print(f"Intercept: {reg_lasso.intercept_:.4f}")

Domyślna regularyzacja jest tak silna, że wszystkie wagi są zerowe, a model zawsze przewiduje tę samą wartość! Jest to skrajne niedouczenie. Zdecydowanie musimy zmniejszyć `alpha`, więc dokonajmy tuningu tego hiperparametru.

## Tuning hiperparametrów

Praktycznie wszystkie modele ML mają hiperparametry, często liczne, które w zauważalny sposób wpływają na wyniki, a szczególnie na niedouczenie i przeuczenie. Ich wartości trzeba zatem dobrać dość dokładnie. Jak to zrobić? Proces doboru hiperparametrów nazywa się **tuningiem hiperparametrów (hyperparameter tuning)**.

Istnieje na to wiele sposobów. Większość z nich polega na tym, że trenuje się za każdym razem model z nowym zestawem hiperparametrów i wybiera się ten zestaw, który pozwala uzyskać najlepsze wyniki. Metody głównie różnią się między sobą sposobem doboru kandydujących zestawów hiperparametrów. Najprostsze i najpopularniejsze to:
* **pełne przeszukiwanie (grid search)** - definiujemy możliwe wartości dla różnych hiperparametrów, a metoda sprawdza wszystkie ich możliwe kombinacje (czyli siatkę)
* **losowe przeszukiwanie (randomized search)** - definiujemy możliwe wartości jak w pełnym przeszukiwaniu, ale sprawdzamy tylko ograniczoną liczbę losowo wybranych kombinacji; jest to przydatne, gdy mamy dużo hiperparametrów

**Uwaga:** warto zauważyć, że liczba możliwych kombinacji rośnie wykładniczo wraz z liczbą hiperparametrów i ich możliwych wartości. Mając siatkę na 3 hiperparametry po 10 możliwych wartości dla każdego, otrzymujemy 1000 możliwych kombinacji. W pracy w ML płacą nam też za to, że wiemy, jakie siatki dobrać :)

Przy tuningu siły regularyzacji często najpierw znajdujemy rząd wielkości, np. za pomocą `np.logspace(-3, 3, 7)`, co daje `[0.001, 0.01, ..., 1000]`. Następnie zagęszczamy siatkę wokół najlepszej wartości.

### Zbiór walidacyjny i walidacja skrośna

Jak ocenić, jak dobry jest jakiś zestaw hiperparametrów? Nie możemy sprawdzić tego na zbiorze treningowym - wyniki byłyby zbyt optymistyczne. Nie możemy wykorzystać zbioru testowego - mielibyśmy wyciek danych (data leakage), bo wybieralibyśmy model explicite pod nasz zbiór testowy. Zbiór testowy musi być całkowicie niezależny od procesu treningu, wyboru i optymalizacji modelu! Trzeba zatem osobnego zbioru, na którym będziemy na bieżąco sprawdzać jakość modeli dla różnych hiperparametrów. Jest to **zbiór walidacyjny (validation set)**.

Zbiór taki wycina się ze zbioru treningowego. Dzielimy zatem nasze dane nie na dwie, ale trzy części: treningową, walidacyjną i testową. Typowe proporcje to 60-20-20% lub 80-10-10%. Taki jednorazowy podział to **holdout**, używamy kiedy mamy dużo danych i 10-20% wystarcza do dobrego oszacowania. Zbyt mały zbiór walidacyjny czy testowy da nam mało wiarygodne wyniki - nie da się nawet powiedzieć, czy zbyt pesymistyczne, czy optymistyczne! W praktyce niestety często mamy mało danych. Trzeba zatem jakiejś metody, która stworzy nam więcej zbiorów walidacyjnych z tej samej ilości danych.

Taką metodą jest **walidacja skrośna (cross-validation, CV)**, a konkretnie *k-fold cross-validation*. Polega na tym, że dzielimy zbiór treningowy na $k$ równych podzbiorów, tzw. *foldów*. Każdy podzbiór po kolei staje się zbiorem walidacyjnym, a pozostałe łączymy w zbiór treningowy. Przykładowo, jeżeli mamy 5 foldów (1, 2, 3, 4, 5), to będziemy mieli po kolei:
- zbiór treningowy: (2, 3, 4, 5), walidacyjny: (1)
- zbiór treningowy: (1, 3, 4, 5), walidacyjny: (2)
- zbiór treningowy: (1, 2, 4, 5), walidacyjny: (3)
- zbiór treningowy: (1, 2, 3, 5), walidacyjny: (4)
- zbiór treningowy: (1, 2, 3, 4), walidacyjny: (5)

Trenujemy zatem $k$ modeli dla tego samego zestawu hiperparametrów i każdy testujemy na jego zbiorze walidacyjnym. Mamy $k$ wyników dla zbiorów walidacyjnych, które możemy uśrednić (i ew. obliczyć odchylenie standardowe). Takie wyniki są znacznie bardziej wiarygodne zgodnie ze statystyką (moc statystyczna itp.). Typowo używa się 5 lub 10 foldów, co jest dobrym balansem między liczbą modeli do wytrenowania i wielkością zbiorów walidacyjnych.

### GridSearchCV

Scikit-learn automatyzuje pełne przeszukiwanie z walidacją skrośną w klasie [GridSearchCV](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html). Istnieje też klasa [RandomizedSearchCV](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.RandomizedSearchCV.html) do losowego przeszukiwania, o podobnym API.

`GridSearchCV` przyjmuje:
- `estimator` - model, którego hiperparametry dobieramy
- `param_grid` - słownik z siatką hiperparametrów, `{"nazwa_hiperparametru": [wartości]}`
- `scoring` - metryka używana do wyboru najlepszego modelu; S\scikit-learn zawsze maksymalizuje wyniki, więc błędy są brane ze znakiem minus, np. `"neg_root_mean_squared_error"`
- `cv` - liczba foldów, domyślnie 5
- `n_jobs` - liczba równoległych procesów; `-1` oznacza użycie wszystkich rdzeni procesora

Używa się go dokładnie tak jak modelu - `.fit()` przeprowadza tuning. Dla kosztownych modeli i dużych siatek może to trwać bardzo długo!

Następnie można użyć:
- `.best_params_` - najlepszy znaleziony zestaw hiperparametrów
- `.best_estimator_` - model z najlepszymi hiperparametrami, wytrenowany ponownie na całym zbiorze treningowym
- `.predict()` - predykcje najlepszego modelu, więc obiektu grid search można używać bezpośrednio jak modelu

Przykład dla regresji ridge:

In [ ]:
from sklearn.model_selection import GridSearchCV

ridge_grid_example = GridSearchCV(
    estimator=Ridge(),  # base model to tune
    param_grid={"alpha": [0.1, 1, 10]},  # hyperparameter grid
    scoring="neg_root_mean_squared_error",  # metric to maximize
    cv=5,
)
ridge_grid_example.fit(X_train, y_train)

print(f"Best hyperparameters: {ridge_grid_example.best_params_}")
evaluate_model(ridge_grid_example, X_train, X_test, y_train, y_test);

### Zadanie 6 (1 punkt)

Dokonaj tuningu hiperparametrów regresji ridge, LASSO i ElasticNet za pomocą `GridSearchCV`. Użyj następujących siatek:
- ridge: `alpha` - 50 wartości z przedziału od $10^{-3}$ do $10^3$, w skali logarytmicznej
- LASSO: `alpha` - 50 wartości z przedziału od $10^{-5}$ do $1$, w skali logarytmicznej
- ElasticNet: `alpha` - tak samo jak dla LASSO, `l1_ratio` - wartości `[0.1, 0.25, 0.5, 0.75, 0.9]`

Dla `Lasso` i `ElasticNet` przekaż `random_state=0` oraz `max_iter=10000`. Dla wszystkich modeli użyj:
- `scoring="neg_root_mean_squared_error"`
- `cv=5`
- `n_jobs=-1`

Zapisz obiekty grid search w zmiennych `ridge_grid`, `lasso_grid` oraz `elastic_net_grid`. Dla każdego modelu wypisz najlepsze hiperparametry i dokonaj jego ewaluacji. Zapisz wyniki w słowniku `results` pod kluczami `"Ridge"`, `"LASSO"` oraz `"ElasticNet"`.

Skomentuj wyniki. Czy regularyzacja zmniejszyła przeuczenie? Który model jest najlepszy?

Przydatna funkcja: `np.logspace()`.

In [ ]:
from sklearn.linear_model import ElasticNet

# your_code

In [ ]:
assert 1 <= ridge_grid.best_params_["alpha"] <= 10
assert lasso_grid.best_params_["alpha"] < 0.01
assert elastic_net_grid.best_params_["alpha"] < 0.01

for name in ["Ridge", "LASSO", "ElasticNet"]:
    assert results[name]["test_rmse"] < 19000
    assert results[name]["test_mae"] < 13000

print("Solution is correct!")

// skomentuj tutaj

### Krzywa walidacji

Dla pojedynczego hiperparametru regularyzacji dobrym sposobem na zobaczenie niedouczenia i przeuczenia jest **krzywa walidacji (validation curve)**. Pokazuje ona błąd treningowy i walidacyjny dla różnych wartości hiperparametru. `GridSearchCV` zapisuje wszystkie wyniki w atrybucie `.cv_results_`, a z `return_train_score=True` zapisuje także wyniki na zbiorach treningowych.

Narysujmy ją dla regresji ridge, dla szerokiego zakresu wartości `alpha`. Zwróć uwagę, że błędy są tutaj wyrażone w logarytmach cen, bo to właśnie bezpośrednio przewiduje model podczas walidacji skrośnej.

In [ ]:
ridge_curve = GridSearchCV(
    estimator=Ridge(),
    param_grid={"alpha": np.logspace(-4, 4, 50)},
    scoring="neg_root_mean_squared_error",
    cv=5,
    n_jobs=-1,
    return_train_score=True,
)
ridge_curve.fit(X_train, y_train)

alphas = ridge_curve.cv_results_["param_alpha"].astype(float)
train_rmse = -ridge_curve.cv_results_["mean_train_score"]
valid_rmse = -ridge_curve.cv_results_["mean_test_score"]
best_alpha = ridge_curve.best_params_["alpha"]

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(alphas, train_rmse, color="blue", lw=2, label="Training")
ax.plot(alphas, valid_rmse, color="orange", lw=2, label="Validation")
ax.axvline(best_alpha, color="gray", ls="--", label=f"Best alpha: {best_alpha:.2f}")
ax.set_xscale("log")
ax.set_title("Validation curve for ridge regression")
ax.set_xlabel("alpha (regularization strength)")
ax.set_ylabel("RMSE (log-price)")
ax.text(0.02, 0.9, "overfitting", transform=ax.transAxes, color="gray")
ax.text(0.8, 0.9, "underfitting", transform=ax.transAxes, color="gray")
ax.legend(loc="center left")
plt.show()

Wyraźnie widać trzy obszary:
- małe `alpha` (po lewej) - błąd treningowy jest najniższy, ale walidacyjny jest wyższy; model jest **przeuczony**
- duże `alpha` (po prawej) - oba błędy szybko rosną i są bardzo blisko siebie; regularyzacja jest zbyt silna i model jest **niedouczony**
- najlepsze `alpha` jest pomiędzy, tam, gdzie błąd walidacyjny jest najniższy

### RidgeCV, LassoCV, ElasticNetCV

W przypadku regresji liniowej istnieją bardzo wydajne implementacje walidacji skrośnej, głównie dzięki prostocie tego modelu. W scikit-learn są to klasy z wbudowanym, wydajnym tuningiem siły regularyzacji: [RidgeCV](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.RidgeCV.html), [LassoCV](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LassoCV.html) oraz [ElasticNetCV](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.ElasticNetCV.html). Używa się ich jak zwykłych modeli, a tuning odbywa się wewnątrz metody `.fit()`.

Są one znacznie szybsze niż `GridSearchCV`, ponieważ:
- `RidgeCV` oblicza rozkład SVD tylko raz, a walidację skrośną przeprowadza za pomocą sprytnie zastosowanej algebry liniowej, wyjaśnionej [w dokumentacji w kodzie](https://github.com/scikit-learn/scikit-learn/blob/8c9c1f27b7e21201cfffb118934999025fd50cca/sklearn/linear_model/_ridge.py#L1547)
- `LassoCV` oraz `ElasticNetCV` iterują od największych do najmniejszych wartości `alpha` (siły regularyzacji), używając rozwiązania dla danej wartości jako punktu początkowego dla kolejnej, czyli po prostu inteligentnie wybierając punkt startowy w optymalizacji funkcji kosztu

Wybrana siła regularyzacji znajduje się w atrybucie `.alpha_`.

### Zadanie 7 (1 punkt)

1. Wytrenuj model `RidgeCV` w zmiennej `reg_ridge_cv`, sprawdzając 100 wartości z przedziału od $10^{-3}$ do $10^3$, w skali logarytmicznej.
2. Wytrenuj model `LassoCV` w zmiennej `reg_lasso_cv`, sprawdzając 100 wartości `alpha` dobranych automatycznie przez scikit-learn (przekaż `alphas=100`). Użyj 5-krotnej walidacji skrośnej, `random_state=0` oraz `max_iter=10000`.
3. Dla porównania uruchom `GridSearchCV` dla `Ridge` z tymi samymi 100 wartościami `alpha` i 5-krotną walidacją skrośną.
4. Zmierz czas treningu wszystkich trzech modeli za pomocą `time.perf_counter()` i go wypisz.
5. Wypisz znalezione siły regularyzacji, dokonaj ewaluacji modeli `RidgeCV` i `LassoCV`, a następnie zapisz ich wyniki w słowniku `results` pod kluczami `"RidgeCV"` oraz `"LassoCV"`.
6. Oblicz, ile cech zostało wyeliminowanych przez LASSO, czyli ma wagi równe zero (użyj `np.isclose()`). Przypisz tę wartość do zmiennej `n_eliminated` i ją wypisz.

Skomentuj wyniki. Czy są podobne do tych z `GridSearchCV`? Czy tuning jest szybszy? Ile cech faktycznie wykorzystuje LASSO?

In [ ]:
import time

from sklearn.linear_model import LassoCV, RidgeCV

# your_code

In [ ]:
assert 1 <= reg_ridge_cv.alpha_ <= 10
assert reg_lasso_cv.alpha_ < 0.01
assert 100 < n_eliminated < 200

assert results["RidgeCV"]["test_rmse"] < 19000
assert results["LassoCV"]["test_rmse"] < 19000

print("Solution is correct!")

// skomentuj tutaj

## Porównanie modeli

Wytrenowaliśmy sporo modeli, więc porównajmy je na jednym wykresie. Ta funkcja przyda się także w drugiej części laboratorium.

In [ ]:
def plot_results(results: dict[str, dict[str, float]], title: str) -> None:
    results_df = pd.DataFrame(results).T

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for ax, metric in zip(axes, ["rmse", "mae"]):
        results_df[[f"train_{metric}", f"test_{metric}"]].plot.bar(
            ax=ax, color=["blue", "orange"], width=0.7, edgecolor="white", linewidth=2
        )
        ax.set_title(f"{title} - {metric.upper()}")
        ax.set_ylabel(metric.upper())
        ax.legend(["Train", "Test"])
        ax.tick_params(axis="x", rotation=30)
    plt.tight_layout()
    plt.show()


plot_results(results, "Ames housing")
pd.DataFrame(results).T.round(2)

## Ważność cech

Modele liniowe są dość łatwe w **interpretacji (interpretability)** i zrozumieniu. Duża waga cechy oznacza, że model jest wrażliwy na jej zmiany, więc ma ona duże znaczenie dla jego predykcji. Co więcej, kierunek zmiany wyznacza znak wagi. Do tego cechy wyeliminowane przez LASSO są prawdopodobnie nieistotne.

Wagi znajdują się w atrybucie `.coef_`, a `.feature_names_in_` zawiera odpowiadające im nazwy cech.

Narysujmy 20 największych co do wartości bezwzględnej wag regresji ridge.

In [ ]:
coefs = pd.Series(reg_ridge_cv.coef_, index=reg_ridge_cv.feature_names_in_)
top_coefs = coefs.loc[coefs.abs().sort_values().index[-20:]]
colors = ["blue" if coef > 0 else "orange" for coef in top_coefs]

ax = top_coefs.plot.barh(figsize=(8, 7), color=colors)
ax.axvline(0, color="gray", lw=1)
ax.set_title("20 largest weights, ridge regression")
ax.set_xlabel("Weight (blue - increases price, orange - decreases price)")
plt.show()

Nasze wagi dotyczą logarytmu ceny, a cechy są w zakresie $[0, 1]$. Przykładowo waga 0.1 oznacza, że cena rośnie około $e^{0.1} \approx 1.105$ razy, czyli o około 10.5%, gdy cecha zmienia się od swojej wartości minimalnej do maksymalnej.

Część z tych cech ma dużo sensu, np. ogólna jakość, powierzchnia mieszkalna czy rok budowy. Kilka jest jednak podejrzanych, np. `Heating_Wall` czy `SaleCondition_AdjLand`. Sprawdźmy, ile domów w zbiorze treningowym faktycznie je ma.

In [ ]:
for col in [
    "Heating_Wall",
    "SaleCondition_AdjLand",
    "SaleCondition_Alloca",
    "MSZoning_C (all)",
]:
    print(f"{col}: {int(X_train[col].sum())} houses")

Tylko kilka domów! Waga pokazuje, jak bardzo zmienia się predykcja wraz ze zmianą cechy, ale nie to, jak często to się zdarza. Duża waga cechy obecnej w 4 domach wpływa tylko na te 4 predykcje.

Wagi trzeba zatem interpretować ostrożnie:
- rzadkie cechy - zanim uznasz wagę cechy za ważną, sprawdź, jak często ta cecha występuje
- skalowanie - przy skalowaniu min-max waga opisuje zmianę od wartości minimalnej do maksymalnej; przy wartościach odstających większość domów znajduje się w małej części tego zakresu (np. `Functional` ma średnią 0.98)
- skorelowane cechy - `GrLivArea` to dokładnie `X1stFlrSF + X2ndFlrSF + LowQualFinSF`, więc model może rozdzielić wagę między nie na wiele sposobów

Biorąc to pod uwagę, najważniejsze cechy z dużymi wagami generalnie mają sens: ogólna jakość (`OverallQual`), powierzchnia mieszkalna (`GrLivArea`), ogólny stan oraz rok budowy zwiększają cenę.

## Zbiór danych Melbourne housing

Teraz Twoja kolej, żeby samodzielnie przeprowadzić cały proces na drugim zbiorze danych. Wykorzystamy zbiór [Melbourne Housing Snapshot](https://www.kaggle.com/datasets/dansbecker/melbourne-housing-snapshot) ([kopia na OpenML](https://www.openml.org/d/46077)), Tony Pino, licencja CC BY-NC-SA 4.0. Zawiera on dane o sprzedaży domów w Melbourne (Australia) z lat 2016-2017, a zadaniem jest przewidywanie ceny w dolarach australijskich (AUD).

Opisy wszystkich kolumn znajdują się w pliku [melbourne_description.txt](melbourne_description.txt), a dane w pliku [melbourne_data.csv](melbourne_data.csv).

Ten zbiór dość znacząco różni się od Ames. Ma znacznie więcej próbek (ponad 13 tysięcy), ale dużo mniej cech, a wiele z nich opisuje lokalizację. Ma też dużo wartości brakujących i obserwacji odstających. Pamiętaj, że używamy tylko funkcji i klas, które już znamy, więc możesz wykorzystać sporo kodu z powyższych zadań.

Ponownie wykorzystamy funkcje `evaluate_model()` i `plot_results()`. Zwróć uwagę, że działają one w oryginalnej jednostce zmiennej docelowej (tutaj AUD), o ile zastosujesz do niej tę samą transformację `np.log1p`.

### Zadanie 8 (1.5 punktu)

1. Wczytaj dane z pliku `melbourne_data.csv` do zmiennej `df_melb`. Wypisz liczbę próbek i cech oraz informacje o kolumnach za pomocą `.info()`.
2. Narysuj procent wartości brakujących dla kolumn z wartościami brakującymi, tak jak w zadaniu 1.
3. Usuń kolumny podane poniżej w `columns_to_drop`. Są to unikalne identyfikatory (`Address`), informacje niebędące cechami samego domu (`SellerG`, czyli agent nieruchomości, oraz `Date`), duplikaty innych cech lokalizacji (`Postcode`) lub kolumny z dużą liczbą wartości brakujących, które też duplikują inne cechy lokalizacji (`CouncilArea`).
4. Narysuj rozkład cen za pomocą `plot_distribution()`. Następnie wyodrębnij kolumnę `Price` do zmiennej `y_melb`, stosując transformację logarytmiczną, i również narysuj jej rozkład.
5. Zapisz listy cech kategorycznych i numerycznych w zmiennych `categorical_features_melb` oraz `numerical_features_melb`.
6. Podziel dane na zbiór treningowy i testowy w proporcjach 75%-25%, z `random_state=0`. Użyj zmiennych `X_melb_train_raw`, `X_melb_test_raw`, `y_melb_train` oraz `y_melb_test`.

In [ ]:
columns_to_drop = ["Address", "SellerG", "Date", "Postcode", "CouncilArea"]

# your_code

In [ ]:
assert df_melb.shape == (13580, 15)
assert "Price" not in df_melb.columns
assert not set(columns_to_drop) & set(df_melb.columns)

assert set(categorical_features_melb) == {"Suburb", "Type", "Method", "Regionname"}
assert len(numerical_features_melb) == 11

assert X_melb_train_raw.shape == (10185, 15)
assert X_melb_test_raw.shape == (3395, 15)
assert 13 < y_melb_train.mean() < 14

print("Solution is correct!")

### Zadanie 9 (2 punkty)

1. Narysuj histogramy cech `Landsize` oraz `BuildingArea` w danych treningowych. Następnie zastosuj dla tych cech winsoryzację do 1. i 99. percentyla, tak jak w zadaniu 3. Zapisz obiekt `Winsorizer` w zmiennej `winsorizer_melb`.
2. Przetwórz cechy numeryczne (imputacja średnią, skalowanie min-max) oraz kategoryczne (one-hot encoding), tak jak w zadaniu 3. Zapisz wyniki w `X_melb_train` oraz `X_melb_test`.
3. Wytrenuj i dokonaj ewaluacji modeli `LinearRegression`, `RidgeCV` oraz `LassoCV`, z tymi samymi hiperparametrami, co w zadaniu 7. Zapisz wyniki w słowniku `results_melb`, pod kluczami `"Linear regression"`, `"RidgeCV"` oraz `"LassoCV"`.
4. Porównaj modele za pomocą `plot_results()`.
5. Narysuj 15 największych co do wartości bezwzględnej wag modelu `RidgeCV`, tak jak w sekcji "Ważność cech". Zapisz wagi w zmiennej `coefs_melb` (jako `pd.Series`, z nazwami cech jako indeksem).

Skomentuj wyniki:
- Czy modele są przeuczone, czy niedouczone? Czy regularyzacja tu pomaga?
- Porównaj RMSE i MAE. Co ta różnica mówi o naszych błędach?
- Które cechy mają największe wagi i w którą stronę wpływają na cenę? Czy ma to sens? Czy są wśród nich jakieś podejrzane cechy, np. rzadkie dzielnice?

In [ ]:
# your_code

In [ ]:
assert X_melb_train.shape == (10185, 330)
assert X_melb_test.shape == (3395, 330)
assert X_melb_train.notna().all().all()
assert X_melb_test.notna().all().all()

assert winsorizer_melb.right_tail_caps_["Landsize"] < 3100
assert winsorizer_melb.right_tail_caps_["BuildingArea"] < 500

for name in ["Linear regression", "RidgeCV", "LassoCV"]:
    assert 340000 < results_melb[name]["test_rmse"] < 370000
    assert 190000 < results_melb[name]["test_mae"] < 210000

assert len(coefs_melb) == 330
assert coefs_melb.abs().idxmax() == "Distance"
assert coefs_melb["Distance"] < 0

print("Solution is correct!")

// skomentuj tutaj

## Zadanie dodatkowe 1 (1 punkt)

Zaimplementuj samodzielnie, zgodnie z interfejsem scikit-learn:
1. Regresję liniową - z użyciem pseudoodwrotności Moore'a-Penrose'a macierzy $X$.
2. Regresję ridge z regularyzacją L2 - z użyciem rozkładu SVD i wzoru dla regularyzacji. Można pokazać, że dla siły regularyzacji $\lambda$ optymalne wagi to:

$$
w = V (\Sigma^2 + I\lambda)^{-1} \Sigma U^T y
$$

Jak uwzględnić wyraz wolny (intercept) $b$:
- regresja liniowa - dodaj do $X$ kolumnę jedynek, a jej waga będzie wyrazem wolnym
- regresja ridge - wyraz wolny nie powinien być regularyzowany, więc nie dodawaj kolumny jedynek. Zamiast tego najpierw wycentruj dane: odejmij średnie kolumn $\bar{x}$ od $X$ oraz średnią $\bar{y}$ od $y$. Oblicz wagi $w$ na wycentrowanych danych za pomocą powyższego wzoru. Wtedy wyraz wolny to $b = \bar{y} - \bar{x}^T w$, dzięki czemu predykcje $\hat{y} = Xw + b$ działają dla oryginalnych, niewycentrowanych danych. Dokładnie tak robi to scikit-learn.

**Nigdy** nie obliczaj explicite $X^TX$, nie jest to potrzebne i byłoby skrajnie nieefektywne.

Interfejs scikit-learn zakłada:
- dziedziczenie po `BaseEstimator` i `RegressorMixin` (lub `ClassifierMixin` dla klasyfikatorów)
- konstruktor:
  - wszystkie argumenty mają wartości domyślne
  - argumenty są zapisywane bez żadnych zmian jako atrybuty o tych samych nazwach
  - jest lekki i nie wykonuje żadnych obliczeń
- metoda `.fit()`:
  - przyjmuje macierz `X` i wektor `y`
  - wywołuje `validate_data()`
  - estymuje wektor wag `w` i wyraz wolny `b`
  - estymowane parametry są tworzone w tej metodzie jako atrybuty z podkreślnikiem na końcu, np. `self.coef_ = ...`
  - zwraca `self`
- metoda `.predict()`:
  - przyjmuje macierz `X`
  - wywołuje `check_is_fitted()` oraz `validate_data()`
  - zwraca predykcje dla tych nowych punktów

Sprawdź poprawność swojej implementacji na zbiorze Ames (`X_train`, `X_test` z powyższych zadań). Wyniki powinny być bardzo bliskie tym ze scikit-learn dla tych samych hiperparametrów.

Przydatne linki:
- [Developing scikit-learn estimators](https://scikit-learn.org/stable/developers/develop.html)
- [SVD in Machine Learning: Ridge Regression and Multicollinearity](https://medium.com/data-science/ridge-regression-and-multicollinearity-d8a3e06efce8)
- [Mathematical Methods in Data Science - Ridge Regression](https://mmids-textbook.github.io/chap04_svd/06_further/roch-mmids-svd-further.html#ridge-regression)
- [Wikipedia - Ridge regression - Relation to singular-value decomposition and Wiener filter](https://en.wikipedia.org/wiki/Ridge_regression#Relation_to_singular-value_decomposition_and_Wiener_filter)

In [ ]:
# your_code

## Zadanie dodatkowe 2 (1 punkt)

Ręczne pisanie transformacji jedna po drugiej wprowadza sporo powtarzalnego kodu (*boilerplate*), np. wywoływanie `.fit()` i `.transform()`, przekazywanie wyników między krokami i powtarzanie wszystkiego dla danych treningowych i testowych. Jest to też ryzykowne: łatwo jest przetworzyć dane treningowe i testowe w niespójny sposób, co nazywa się **train-test skew**.

Scikit-learn rozwiązuje ten problem za pomocą pipeline'ów:
- [Pipeline](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html) - wykonuje kroki sekwencyjnie, np. imputację, a następnie skalowanie; ostatnim krokiem może być model ML
- [ColumnTransformer](https://scikit-learn.org/stable/modules/generated/sklearn.compose.ColumnTransformer.html) - wykonuje transformacje równolegle, dla różnych zbiorów kolumn, np. numerycznych i kategorycznych

Całego pipeline'u używa się jak pojedynczego modelu, z jednym wywołaniem `.fit()` i `.predict()`, przyjmując surowy `DataFrame` jako wejście. Bardzo przydatne jest to, że można łatwo podmieniać poszczególne kroki, używając `.set_params()` z nazwami kroków oddzielonymi podwójnym podkreślnikiem, np. `pipeline.set_params(preprocessing__numerical__scaler=StandardScaler())`.

Transformacje z Feature-engine działają również z pipeline'ami scikit-learn. Przyjmują i zwracają `DataFrame`'y, a kolumny do przekształcenia wybiera się parametrem `variables`.

1. Stwórz pojedynczy obiekt `Pipeline` dla zbioru Ames, który przyjmuje surowe dane (`X_train_raw` i `X_test_raw`) i zawiera wszystkie kroki przetwarzania z zadania 3 oraz model `RidgeCV` z zadania 7. Użyj `Winsorizer` z tymi samymi parametrami, co w zadaniu 3.
2. Sprawdź, czy wyniki są takie same, jak w zadaniu 7.
3. Porównaj różne opcje przetwarzania danych, podmieniając kroki pipeline'u za pomocą `.set_params()`:
    - skalowanie za pomocą `MinMaxScaler`, `StandardScaler` oraz `RobustScaler`,
    - imputację średnią i medianą.

   Narysuj wyniki za pomocą `plot_results()`.

Skomentuj wyniki. Które przetwarzanie działa najlepiej? Czy ma to duże znaczenie dla tego zbioru?

In [ ]:
# your_code